# Active Learning Backend Test Notebook

This notebook tests all backend components before building the Streamlit dashboard.

## Test Coverage
- **Task 1**: Backend extensions (Commands, Probe Images, Confusion Matrix)
- **Task 2**: Worker process functions
- **Task 3**: StateManager integration

Run each cell sequentially and verify the outputs match expected behavior.

## 0. Setup and Imports

In [1]:
import sys
import os
import json
import shutil
import numpy as np
from pathlib import Path
from datetime import datetime
import torch
import timm


# Core imports
from src.config import Config
from src.state import (
    StateManager,
    ExperimentManager,
    ExperimentState,
    ExperimentPhase,
    Command,
    EpochMetrics,
    CycleMetrics,
    QueriedImage,
    ProbeImage,
    AnnotationSubmission,
    UserAnnotation,
    create_experiment_config_from_config
)
from src.dataloader import get_datasets, get_dataset_info
from src.models import get_model, get_model_info
from src.trainer import Trainer
from src.data_manager import ALDataManager
from src.active_loop import ActiveLearningLoop
from src.strategies import get_strategy

print("✅ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful
PyTorch version: 2.3.1+cu121
CUDA available: True
GPU: NVIDIA H100 PCIe MIG 3g.40gb


/usr/local/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_probabilities" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [2]:
# Configuration for tests
TEST_EXPERIMENT_ID = "test_backend_" + datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENTS_DIR = Path("experiments")
TEST_EXP_DIR = EXPERIMENTS_DIR / TEST_EXPERIMENT_ID

# Data directory - adjust if needed
DATA_DIR = "./data/raw/kaggle-vehicle/"

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Test Experiment ID: {TEST_EXPERIMENT_ID}")
print(f"Experiment Directory: {TEST_EXP_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Device: {DEVICE}")

Test Experiment ID: test_backend_20251211_164729
Experiment Directory: experiments/test_backend_20251211_164729
Data Directory: ./data/raw/kaggle-vehicle/
Device: cuda


In [3]:
# Verify dataset exists
data_path = Path(DATA_DIR)
if not data_path.exists():
    print(f"❌ Data directory not found: {DATA_DIR}")
    print("Please update DATA_DIR to point to your Kaggle vehicle dataset")
else:
    dataset_info = get_dataset_info(DATA_DIR)
    print(f"✅ Dataset found")
    print(f"   Total images: {dataset_info['total_images']}")
    print(f"   Classes: {dataset_info['class_names']}")
    print(f"   Distribution: {dataset_info['class_counts']}")

✅ Dataset found
   Total images: 400
   Classes: ['bus', 'car', 'motorcycle', 'truck']
   Distribution: {'bus': 100, 'car': 100, 'motorcycle': 100, 'truck': 100}


---
## 1. Test Configuration and State Initialization

Tests:
- Config creation and serialization
- ExperimentManager directory creation
- StateManager initialization

In [4]:
# Create test configuration
config = Config()

# Model config
config.model.name = "resnet18"
config.model.pretrained = True
config.model.num_classes = 4

# Training config - reduced for testing
config.training.epochs = 3  # Small number for quick testing
config.training.batch_size = 16
config.training.learning_rate = 0.001
config.training.optimizer = "adam"
config.training.early_stopping_patience = 5
config.training.seed = 42

# Data config
config.data.data_dir = DATA_DIR
config.data.val_split = 0.15
config.data.test_split = 0.15
config.data.augmentation = True
config.data.num_workers = 2

# Active Learning config
config.active_learning.enabled = True
config.active_learning.num_cycles = 2  # Small for testing
config.active_learning.sampling_strategy = "uncertainty"
config.active_learning.uncertainty_method = "least_confidence"
config.active_learning.initial_pool_size = 20  # Small for testing
config.active_learning.batch_size_al = 10
config.active_learning.reset_mode = "pretrained"

print("✅ Config created")
print(f"   Model: {config.model.name}")
print(f"   Strategy: {config.active_learning.sampling_strategy}")
print(f"   Cycles: {config.active_learning.num_cycles}")
print(f"   Initial pool: {config.active_learning.initial_pool_size}")
print(f"   Query batch: {config.active_learning.batch_size_al}")

✅ Config created
   Model: resnet18
   Strategy: uncertainty
   Cycles: 2
   Initial pool: 20
   Query batch: 10


In [5]:
# Test ExperimentManager
exp_manager = ExperimentManager(EXPERIMENTS_DIR)

# Create experiment directory
exp_dir = exp_manager.create_experiment(
    name="backend_test",
    experiment_id=TEST_EXPERIMENT_ID
)

print(f"✅ Experiment directory created: {exp_dir}")
print(f"   Subdirectories: {[d.name for d in exp_dir.iterdir() if d.is_dir()]}")

# Save config
config.save_to(str(exp_dir / "config.yaml"))
print(f"✅ Config saved to {exp_dir / 'config.yaml'}")

✅ Experiment directory created: experiments/test_backend_20251211_164729
   Subdirectories: ['probes', 'queries', 'checkpoints']
✅ Config saved to experiments/test_backend_20251211_164729/config.yaml


In [6]:
# Test StateManager initialization
state_manager = StateManager(exp_dir)

# Initialize experiment state
state = state_manager.initialize_state(
    experiment_id=TEST_EXPERIMENT_ID,
    experiment_name="Backend Test Experiment"
)

print(f"✅ State initialized")
print(f"   Experiment ID: {state.experiment_id}")
print(f"   Phase: {state.phase}")
print(f"   State file exists: {state_manager.state_exists()}")

✅ State initialized
   Experiment ID: test_backend_20251211_164729
   Phase: IDLE
   State file exists: True


In [7]:
# Add experiment config to state
exp_config = create_experiment_config_from_config(config)
exp_config.class_names = dataset_info['class_names']

state_manager.update_state(
    config=exp_config,
    total_cycles=config.active_learning.num_cycles
)

# Verify state
state = state_manager.read_state()
print(f"✅ Config added to state")
print(f"   Model: {state.config.model_name}")
print(f"   Strategy: {state.config.sampling_strategy}")
print(f"   Total cycles: {state.total_cycles}")

✅ Config added to state
   Model: resnet18
   Strategy: uncertainty
   Total cycles: 2


---
## 2. Test Command Infrastructure (Task 1.1)

Tests:
- Command enum values
- set_command() and clear_command()
- Command persistence in state file

In [8]:
# Test Command enum
print("Command enum values:")
for cmd in Command:
    print(f"   {cmd.name} = {cmd.value}")

assert Command.START_CYCLE.value == "START_CYCLE"
assert Command.PAUSE.value == "PAUSE"
assert Command.STOP.value == "STOP"
assert Command.CONTINUE.value == "CONTINUE"
print("\n✅ Command enum verified")

Command enum values:
   START_CYCLE = START_CYCLE
   PAUSE = PAUSE
   STOP = STOP
   CONTINUE = CONTINUE

✅ Command enum verified


In [9]:
# Test set_command
state_manager.set_command(Command.START_CYCLE)
state = state_manager.read_state()
assert state.command == Command.START_CYCLE, f"Expected START_CYCLE, got {state.command}"
print(f"✅ set_command works: {state.command}")

# Test clear_command
state_manager.clear_command()
state = state_manager.read_state()
assert state.command is None, f"Expected None, got {state.command}"
print(f"✅ clear_command works: command is None")

# Test all commands
for cmd in [Command.PAUSE, Command.STOP, Command.CONTINUE]:
    state_manager.set_command(cmd)
    state = state_manager.read_state()
    assert state.command == cmd
    state_manager.clear_command()
print("✅ All commands work correctly")

✅ set_command works: START_CYCLE
✅ clear_command works: command is None
✅ All commands work correctly


---
## 3. Test Data Loading and AL Components

Tests:
- Dataset loading with splits
- ALDataManager initialization
- Model loading
- Trainer initialization

In [10]:
# Load datasets
datasets = get_datasets(
    data_dir=config.data.data_dir,
    val_split=config.data.val_split,
    test_split=config.data.test_split,
    augmentation=config.data.augmentation,
    seed=config.training.seed
)

print("✅ Datasets loaded")
print(f"   Train: {len(datasets['train_dataset'])} samples")
print(f"   Val: {len(datasets['val_dataset'])} samples")
print(f"   Test: {len(datasets['test_dataset'])} samples")
print(f"   Classes: {datasets['class_names']}")

✅ Datasets loaded
   Train: 280 samples
   Val: 60 samples
   Test: 60 samples
   Classes: ['bus', 'car', 'motorcycle', 'truck']


In [11]:
# Create DataLoaders for val and test (fixed throughout AL)
from torch.utils.data import DataLoader

val_loader = DataLoader(
    datasets["val_dataset"],
    batch_size=config.training.batch_size,
    shuffle=False,
    num_workers=config.data.num_workers
)

test_loader = DataLoader(
    datasets["test_dataset"],
    batch_size=config.training.batch_size,
    shuffle=False,
    num_workers=config.data.num_workers
)

print(f"✅ DataLoaders created")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

✅ DataLoaders created
   Val batches: 4
   Test batches: 4


In [12]:
# Initialize ALDataManager
data_manager = ALDataManager(
    dataset=datasets["train_dataset"],
    initial_pool_size=config.active_learning.initial_pool_size,
    seed=config.training.seed,
    exp_dir=exp_dir
)

pool_info = data_manager.get_pool_info()
print("✅ ALDataManager initialized")
print(f"   Total: {pool_info['total']}")
print(f"   Labeled: {pool_info['labeled']} ({pool_info['labeled_percentage']:.1f}%)")
print(f"   Unlabeled: {pool_info['unlabeled']} ({pool_info['unlabeled_percentage']:.1f}%)")

✅ ALDataManager initialized
   Total: 280
   Labeled: 20 (7.1%)
   Unlabeled: 260 (92.9%)


In [13]:
# Load model
model = get_model(config.model, device=DEVICE)
model_info = get_model_info(model)

print("✅ Model loaded")
print(f"   Architecture: {config.model.name}")
print(f"   Total params: {model_info['total_parameters']:,}")
print(f"   Trainable params: {model_info['trainable_parameters']:,}")

✅ Model loaded
   Architecture: resnet18
   Total params: 11,178,564
   Trainable params: 11,178,564


In [14]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    config=config,
    exp_dir=exp_dir,
    device=DEVICE
)

print("✅ Trainer initialized")
print(f"   Device: {trainer.device}")
print(f"   Checkpoint dir: {trainer.checkpoint_dir}")

✅ Trainer initialized
   Device: cuda
   Checkpoint dir: experiments/test_backend_20251211_164729/checkpoints


In [15]:
# Get sampling strategy
strategy = get_strategy(
    config.active_learning.sampling_strategy,
    config.active_learning.uncertainty_method
)

print(f"✅ Strategy loaded: {strategy.__name__}")

✅ Strategy loaded: uncertainty_least_confidence


In [16]:
# Initialize ActiveLearningLoop
al_loop = ActiveLearningLoop(
    trainer=trainer,
    data_manager=data_manager,
    strategy=strategy,
    val_loader=val_loader,
    test_loader=test_loader,
    exp_dir=exp_dir,
    config=config,
    class_names=datasets['class_names']
)

print("✅ ActiveLearningLoop initialized")
print(f"   Cycles configured: {config.active_learning.num_cycles}")
print(f"   Current cycle: {al_loop.current_cycle}")

✅ ActiveLearningLoop initialized
   Cycles configured: 2
   Current cycle: 0


---
## 4. Test Probe Image Initialization (Task 1.2)

Tests:
- `_initialize_probe_images()` method
- Stratified selection across classes
- ProbeImage data structure

In [17]:
# Test probe image initialization
probe_images = al_loop._initialize_probe_images(n_probes=12)

print(f"✅ Probe images initialized: {len(probe_images)}")
print(f"\nProbe image details:")

# Check class distribution
class_counts = {}
for probe in probe_images:
    cls = probe.true_class
    class_counts[cls] = class_counts.get(cls, 0) + 1
    
print(f"\nClass distribution in probes:")
for cls, count in sorted(class_counts.items()):
    print(f"   {cls}: {count}")

# Verify probe structure
sample_probe = probe_images[0]
print(f"\nSample probe structure:")
print(f"   image_id: {sample_probe.image_id}")
print(f"   true_class: {sample_probe.true_class}")
print(f"   true_class_idx: {sample_probe.true_class_idx}")
print(f"   probe_type: {sample_probe.probe_type}")
print(f"   predictions_by_cycle: {sample_probe.predictions_by_cycle}")

/usr/local/lib/python3.10/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ Probe images initialized: 8

Probe image details:

Class distribution in probes:
   bus: 3
   car: 2
   motorcycle: 2
   truck: 1

Sample probe structure:
   image_id: 24
   true_class: bus
   true_class_idx: 0
   probe_type: validation_stratified
   predictions_by_cycle: {}


In [18]:
# Update state with probe images
state_manager.update_probe_images(probe_images)

# Verify in state
state = state_manager.read_state()
print(f"✅ Probe images saved to state: {len(state.probe_images)} probes")

✅ Probe images saved to state: 8 probes


---
## 5. Test Single Cycle Execution (Task 2.3)

This simulates what the worker does when it receives START_CYCLE.

Tests:
- `prepare_cycle()`
- `train_single_epoch()` with state updates
- `run_evaluation()` with confusion matrix saving
- `_update_probe_predictions()`
- `query_samples()`

In [19]:
# Phase 1: PREPARING
print("=" * 60)
print("PHASE 1: PREPARING")
print("=" * 60)

state_manager.update_state(phase=ExperimentPhase.PREPARING)

cycle_num = 1
cycle_info = al_loop.prepare_cycle(cycle_num)

print(f"✅ Cycle {cycle_num} prepared")
print(f"   Labeled: {cycle_info['labeled_count']}")
print(f"   Unlabeled: {cycle_info['unlabeled_count']}")
print(f"   Reset mode: {cycle_info['reset_mode']}")
print(f"   Train batches: {cycle_info['train_batches']}")

# Update state
state_manager.update_state(
    current_cycle=cycle_num,
    labeled_count=cycle_info['labeled_count'],
    unlabeled_count=cycle_info['unlabeled_count'],
    current_cycle_epochs=[]
)

PHASE 1: PREPARING
✅ Cycle 1 prepared
   Labeled: 20
   Unlabeled: 260
   Reset mode: pretrained
   Train batches: 2


ExperimentState(experiment_id='test_backend_20251211_164729', experiment_name='Backend Test Experiment', created_at=datetime.datetime(2025, 12, 11, 16, 47, 29, 918783), phase=<ExperimentPhase.PREPARING: 'PREPARING'>, command=None, worker_pid=None, last_heartbeat=None, error_message=None, config=ExperimentConfig(model_name='resnet18', pretrained=True, num_classes=4, class_names=['bus', 'car', 'motorcycle', 'truck'], epochs_per_cycle=3, batch_size=16, learning_rate=0.001, optimizer='adam', weight_decay=0.0001, early_stopping_patience=5, num_cycles=2, sampling_strategy='uncertainty', uncertainty_method='least_confidence', initial_pool_size=20, batch_size_al=10, reset_mode='pretrained', seed=42, data_dir='./data/raw/kaggle-vehicle/', val_split=0.15, test_split=0.15, augmentation=True), current_cycle=1, total_cycles=2, current_epoch=0, epochs_in_cycle=0, labeled_count=20, unlabeled_count=260, total_train_samples=0, cycle_results=[], current_cycle_epochs=[], queried_images=[], probe_images=[

In [20]:
# Phase 2: TRAINING
print("\n" + "=" * 60)
print("PHASE 2: TRAINING")
print("=" * 60)

state_manager.update_state(phase=ExperimentPhase.TRAINING)

epochs = config.training.epochs
print(f"Training for {epochs} epochs...\n")

for epoch in range(1, epochs + 1):
    # Train one epoch
    epoch_metrics = al_loop.train_single_epoch(epoch)
    
    # Update state (simulating worker behavior)
    state_manager.add_epoch_metrics(epoch_metrics)
    
    # Log progress
    val_str = f", Val Acc: {epoch_metrics.val_accuracy:.4f}" if epoch_metrics.val_accuracy else ""
    print(f"   Epoch {epoch}/{epochs} | "
          f"Train Loss: {epoch_metrics.train_loss:.4f}, "
          f"Train Acc: {epoch_metrics.train_accuracy:.4f}{val_str}")
    
    # Check early stopping
    if al_loop.should_stop_early():
        print(f"   Early stopping triggered at epoch {epoch}")
        break

# Verify state updates
state = state_manager.read_state()
print(f"\n✅ Training complete")
print(f"   Epochs in state: {len(state.current_cycle_epochs)}")
print(f"   Current epoch: {state.current_epoch}")


PHASE 2: TRAINING
Training for 3 epochs...



/usr/local/lib/python3.10/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


   Epoch 1/3 | Train Loss: 1.3898, Train Acc: 0.1500, Val Acc: 0.3167


/usr/local/lib/python3.10/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


   Epoch 2/3 | Train Loss: 1.3599, Train Acc: 0.5500, Val Acc: 0.2833


/usr/local/lib/python3.10/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


   Epoch 3/3 | Train Loss: 1.2491, Train Acc: 0.6500, Val Acc: 0.4000

✅ Training complete
   Epochs in state: 3
   Current epoch: 3


In [21]:
# Phase 3: EVALUATING
print("\n" + "=" * 60)
print("PHASE 3: EVALUATING")
print("=" * 60)

state_manager.update_state(phase=ExperimentPhase.EVALUATING)

# Test confusion matrix saving (Task 1.4)
cm_path = exp_dir / f"cycle_{cycle_num}_confusion_matrix.npy"

test_metrics = trainer.evaluate(
    test_loader,
    class_names=datasets['class_names'],
    save_cm_path=cm_path
)

print(f"✅ Evaluation complete")
print(f"   Test Accuracy: {test_metrics['test_accuracy']:.4f}")
print(f"   Test F1: {test_metrics['test_f1']:.4f}")
print(f"   Test Precision: {test_metrics['test_precision']:.4f}")
print(f"   Test Recall: {test_metrics['test_recall']:.4f}")

# Verify confusion matrix saved
if cm_path.exists():
    cm = np.load(cm_path)
    print(f"\n✅ Confusion matrix saved: {cm_path}")
    print(f"   Shape: {cm.shape}")
    print(f"   Matrix:\n{cm}")
else:
    print(f"\n❌ Confusion matrix not saved!")


PHASE 3: EVALUATING
✅ Evaluation complete
   Test Accuracy: 0.3833
   Test F1: 0.3223
   Test Precision: 0.5282
   Test Recall: 0.3833

✅ Confusion matrix saved: experiments/test_backend_20251211_164729/cycle_1_confusion_matrix.npy
   Shape: (4, 4)
   Matrix:
[[ 3  8  2  1]
 [ 0 15  0  0]
 [ 0 10  3  0]
 [ 3 13  0  2]]


In [22]:

# Test probe prediction updates (Task 1.3)
print("\nUpdating probe predictions...")
al_loop._update_probe_predictions(cycle_num)

# Update state with new probe predictions
state_manager.update_probe_images(al_loop.probe_images)

# Verify predictions
print(f"\n✅ Probe predictions updated for cycle {cycle_num}")
for i, probe in enumerate(al_loop.probe_images[:3]):  # Show first 3
    if cycle_num in probe.predictions_by_cycle:
        pred = probe.predictions_by_cycle[cycle_num]
        correct = "✓" if pred['predicted_class'] == probe.true_class else "✗"
        print(f"   Probe {i}: True={probe.true_class}, "
              f"Pred={pred['predicted_class']} ({pred['confidence']:.2%}) {correct}")


Updating probe predictions...

✅ Probe predictions updated for cycle 1
   Probe 0: True=bus, Pred=car (29.85%) ✗
   Probe 1: True=bus, Pred=bus (29.53%) ✓
   Probe 2: True=bus, Pred=bus (28.82%) ✓


In [23]:
# Finalize cycle metrics
cycle_metrics = al_loop.finalize_cycle(test_metrics)
state_manager.finalize_cycle(cycle_metrics)

print(f"\n✅ Cycle {cycle_num} metrics finalized")
print(f"   Labeled pool: {cycle_metrics.labeled_pool_size}")
print(f"   Best val accuracy: {cycle_metrics.best_val_accuracy:.4f}")
print(f"   Test accuracy: {cycle_metrics.test_accuracy:.4f}")


✅ Cycle 1 metrics finalized
   Labeled pool: 20
   Best val accuracy: 0.4000
   Test accuracy: 0.3833


In [24]:
# Phase 4: QUERYING
print("\n" + "=" * 60)
print("PHASE 4: QUERYING")
print("=" * 60)

state_manager.update_state(phase=ExperimentPhase.QUERYING)

queried_images = al_loop.query_samples()

print(f"✅ Queried {len(queried_images)} samples")
print(f"\nQueried image details:")
for i, img in enumerate(queried_images[:5]):  # Show first 5
    print(f"   [{i}] ID={img.image_id}, "
          f"Pred={img.predicted_class} ({img.predicted_confidence:.2%}), "
          f"Uncertainty={img.uncertainty_score:.4f}")
    print(f"       Reason: {img.selection_reason}")
    print(f"       GT: {img.ground_truth_name}")


PHASE 4: QUERYING
✅ Queried 10 samples

Queried image details:
   [0] ID=246, Pred=car (29.49%), Uncertainty=0.7051
       Reason: Low confidence: 29%
       GT: truck
   [1] ID=72, Pred=bus (26.30%), Uncertainty=0.7370
       Reason: Low confidence: 26%
       GT: bus
   [2] ID=58, Pred=car (28.12%), Uncertainty=0.7188
       Reason: Low confidence: 28%
       GT: bus
   [3] ID=194, Pred=car (36.55%), Uncertainty=0.6345
       Reason: Low confidence: 37%
       GT: truck
   [4] ID=270, Pred=car (27.14%), Uncertainty=0.7286
       Reason: Low confidence: 27%
       GT: bus


In [25]:
# Update state with queried images
state_manager.set_queried_images(queried_images)

# Verify state
state = state_manager.read_state()
print(f"\n✅ State updated")
print(f"   Phase: {state.phase}")
print(f"   Queried images in state: {len(state.queried_images)}")


✅ State updated
   Phase: AWAITING_ANNOTATION
   Queried images in state: 10


In [26]:
# Phase 5: AWAITING_ANNOTATION
print("\n" + "=" * 60)
print("PHASE 5: AWAITING_ANNOTATION")
print("=" * 60)

state_manager.update_state(phase=ExperimentPhase.AWAITING_ANNOTATION)

state = state_manager.read_state()
print(f"✅ State set to AWAITING_ANNOTATION")
print(f"   Phase: {state.phase}")
print(f"   Worker would now wait for annotations...")


PHASE 5: AWAITING_ANNOTATION
✅ State set to AWAITING_ANNOTATION
   Phase: AWAITING_ANNOTATION
   Worker would now wait for annotations...


---
## 6. Test Annotation Submission Flow

Tests:
- `write_annotations()` - simulating dashboard submission
- `read_annotations()` - simulating worker reading
- `receive_annotations()` - processing annotations
- Pool updates after annotation

In [27]:
# Simulate user annotations (using ground truth for simulated mode)
user_annotations = []
for img in queried_images:
    user_annotations.append(UserAnnotation(
        image_id=img.image_id,
        user_label=img.ground_truth,  # Using ground truth
        user_label_name=img.ground_truth_name,
        timestamp=datetime.now(),
        was_correct=True  # Since we're using ground truth
    ))

# Create annotation submission
submission = AnnotationSubmission(
    experiment_id=TEST_EXPERIMENT_ID,
    cycle=cycle_num,
    annotations=user_annotations,
    submitted_at=datetime.now()
)

# Write annotations (simulating dashboard)
state_manager.write_annotations(submission)

print(f"✅ Annotations submitted")
print(f"   Count: {len(user_annotations)}")
print(f"   Annotations file exists: {state_manager.annotations_pending()}")

✅ Annotations submitted
   Count: 10
   Annotations file exists: True


In [28]:
# Simulate worker reading annotations
read_annotations = state_manager.read_annotations()

print(f"✅ Annotations read by worker")
print(f"   Experiment: {read_annotations.experiment_id}")
print(f"   Cycle: {read_annotations.cycle}")
print(f"   Count: {len(read_annotations.annotations)}")

✅ Annotations read by worker
   Experiment: test_backend_20251211_164729
   Cycle: 1
   Count: 10


In [29]:
# Process annotations
annotation_list = [
    {"image_id": ann.image_id, "user_label": ann.user_label}
    for ann in read_annotations.annotations
]

result = al_loop.receive_annotations(annotation_list)

print(f"✅ Annotations processed")
print(f"   Moved to labeled: {result['moved_count']}")
print(f"   Annotation accuracy: {result['annotation_accuracy']:.2%}")

# Check updated pool
pool_info = data_manager.get_pool_info()
print(f"\nUpdated pool:")
print(f"   Labeled: {pool_info['labeled']}")
print(f"   Unlabeled: {pool_info['unlabeled']}")

✅ Annotations processed
   Moved to labeled: 10
   Annotation accuracy: 100.00%

Updated pool:
   Labeled: 30
   Unlabeled: 250


In [30]:
# Clear annotations file
state_manager.clear_annotations()

print(f"✅ Annotations cleared")
print(f"   Annotations pending: {state_manager.annotations_pending()}")

# Set state to IDLE (ready for next cycle)
state_manager.update_state(phase=ExperimentPhase.IDLE)
print(f"   Phase set to: IDLE")

✅ Annotations cleared
   Annotations pending: False
   Phase set to: IDLE


---
## 7. Test State Persistence (Task 2.4)

Tests:
- Pool state saving
- Model checkpoint saving
- State file integrity

In [31]:
# Save pool state
pool_state_file = exp_dir / f"cycle_{cycle_num}_pool_state.json"
data_manager.save_state(pool_state_file)

print(f"✅ Pool state saved: {pool_state_file}")

# Verify contents
with open(pool_state_file) as f:
    pool_state = json.load(f)
    
print(f"   Labeled indices: {len(pool_state['labeled_indices'])}")
print(f"   Unlabeled indices: {len(pool_state['unlabeled_indices'])}")
print(f"   Query history entries: {len(pool_state['query_history'])}")

✅ Pool state saved: experiments/test_backend_20251211_164729/cycle_1_pool_state.json
   Labeled indices: 30
   Unlabeled indices: 250
   Query history entries: 1


In [32]:
# Save model checkpoint
trainer.save_cycle_checkpoint(cycle_num)

checkpoint_path = exp_dir / "checkpoints" / f"best_model_cycle_{cycle_num}.pth"
print(f"✅ Checkpoint saved: {checkpoint_path}")
print(f"   Exists: {checkpoint_path.exists()}")

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    print(f"   Keys: {list(checkpoint.keys())}")

✅ Checkpoint saved: experiments/test_backend_20251211_164729/checkpoints/best_model_cycle_1.pth
   Exists: True
   Keys: ['cycle', 'model_state_dict', 'optimizer_state_dict', 'best_val_accuracy', 'best_epoch']


In [33]:
# Verify complete state file
state = state_manager.read_state()

print("✅ Final state verification")
print(f"   Experiment ID: {state.experiment_id}")
print(f"   Phase: {state.phase}")
print(f"   Current cycle: {state.current_cycle}")
print(f"   Cycle results: {len(state.cycle_results)}")
print(f"   Probe images: {len(state.probe_images)}")

if state.cycle_results:
    last_cycle = state.cycle_results[-1]
    print(f"\nLast cycle metrics:")
    print(f"   Cycle: {last_cycle.cycle}")
    print(f"   Test accuracy: {last_cycle.test_accuracy:.4f}")
    print(f"   Labeled pool: {last_cycle.labeled_pool_size}")

✅ Final state verification
   Experiment ID: test_backend_20251211_164729
   Phase: IDLE
   Current cycle: 1
   Cycle results: 1
   Probe images: 8

Last cycle metrics:
   Cycle: 1
   Test accuracy: 0.3833
   Labeled pool: 20


---
## 8. Test Resume Detection (Task 3.3)

Tests:
- Incomplete experiment detection
- Experiment listing

In [34]:
# List all experiments
experiments = exp_manager.list_experiments()

print(f"✅ Found {len(experiments)} experiments")
for exp in experiments:
    print(f"   {exp['experiment_id']}: phase={exp.get('phase', 'N/A')}")

✅ Found 18 experiments
   al_pretrained_test_uncertainty_20251204_141710: phase=None
   baseline_test_20251119_160618: phase=None
   baseline_test_20251119_174543: phase=None
   exp_001: phase=None
   resnet50_test_20251121_095455: phase=None
   test_al_loop: phase=None
   test_al_loop_multiple: phase=None
   test_al_loop_results: phase=None
   test_al_loop_single: phase=None
   test_al_strategy_entropy: phase=None
   test_al_strategy_margin: phase=None
   test_al_strategy_random: phase=None
   test_al_strategy_uncertainty: phase=None
   test_backend_20251211_161333: phase=IDLE
   test_backend_20251211_163432: phase=EVALUATING
   test_backend_20251211_164350: phase=EVALUATING
   test_backend_20251211_164729: phase=IDLE
   test_data_manager: phase=None


In [35]:
# Check if our test experiment is incomplete (should be - only 1 of 2 cycles done)
state = state_manager.read_state()
is_incomplete = state.current_cycle < state.total_cycles

print(f"\nIncomplete experiment check:")
print(f"   Current cycle: {state.current_cycle}")
print(f"   Total cycles: {state.total_cycles}")
print(f"   Is incomplete: {is_incomplete}")

if is_incomplete:
    print(f"   → This experiment can be resumed!")


Incomplete experiment check:
   Current cycle: 1
   Total cycles: 2
   Is incomplete: True
   → This experiment can be resumed!


---
## 9. Test Cycle 2 (Optional)

Run another cycle to verify the full loop works across multiple cycles.

In [36]:
# Run cycle 2 to verify multi-cycle operation
RUN_CYCLE_2 = True  # Set to False to skip

if RUN_CYCLE_2:
    print("=" * 60)
    print("RUNNING CYCLE 2")
    print("=" * 60)
    
    cycle_num = 2
    
    # Prepare
    cycle_info = al_loop.prepare_cycle(cycle_num)
    print(f"\nPrepared cycle {cycle_num}")
    print(f"   Labeled: {cycle_info['labeled_count']} (was {config.active_learning.initial_pool_size})")
    
    state_manager.update_state(
        phase=ExperimentPhase.TRAINING,
        current_cycle=cycle_num,
        labeled_count=cycle_info['labeled_count'],
        unlabeled_count=cycle_info['unlabeled_count'],
        current_cycle_epochs=[]
    )
    
    # Train
    for epoch in range(1, config.training.epochs + 1):
        metrics = al_loop.train_single_epoch(epoch)
        state_manager.add_epoch_metrics(metrics)
        print(f"   Epoch {epoch}: Train Acc={metrics.train_accuracy:.4f}, Val Acc={metrics.val_accuracy:.4f}")
    
    # Evaluate
    cm_path = exp_dir / f"cycle_{cycle_num}_confusion_matrix.npy"
    test_metrics = trainer.evaluate(test_loader, datasets['class_names'], cm_path)
    print(f"\n   Test Accuracy: {test_metrics['test_accuracy']:.4f}")
    
    # Update probes
    al_loop._update_probe_predictions(cycle_num)
    state_manager.update_probe_images(al_loop.probe_images)
    
    # Finalize
    cycle_metrics = al_loop.finalize_cycle(test_metrics)
    state_manager.finalize_cycle(cycle_metrics)
    
    # Check if experiment is now complete
    state = state_manager.read_state()
    if state.current_cycle >= state.total_cycles:
        state_manager.update_state(phase=ExperimentPhase.COMPLETED)
        print(f"\n✅ Experiment COMPLETED!")
    
    print(f"\nCycle results:")
    for cr in state.cycle_results:
        print(f"   Cycle {cr.cycle}: Test Acc={cr.test_accuracy:.4f}, Labeled={cr.labeled_pool_size}")

RUNNING CYCLE 2

Prepared cycle 2
   Labeled: 30 (was 20)


/usr/local/lib/python3.10/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


   Epoch 1: Train Acc=0.2333, Val Acc=0.5833


/usr/local/lib/python3.10/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


   Epoch 2: Train Acc=0.6333, Val Acc=0.4500


/usr/local/lib/python3.10/site-packages/PIL/Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


   Epoch 3: Train Acc=0.7333, Val Acc=0.4667

   Test Accuracy: 0.3833

✅ Experiment COMPLETED!

Cycle results:
   Cycle 1: Test Acc=0.3833, Labeled=20
   Cycle 2: Test Acc=0.3833, Labeled=30


---
## 10. Test Probe Prediction Evolution

Verify predictions are tracked across cycles.

In [37]:
# Check probe prediction evolution
state = state_manager.read_state()

print("Probe Prediction Evolution")
print("=" * 60)

for i, probe in enumerate(state.probe_images[:5]):  # First 5 probes
    print(f"\nProbe {i}: True class = {probe.true_class}")
    
    for cycle, pred in sorted(probe.predictions_by_cycle.items()):
        correct = "✓" if pred['predicted_class'] == probe.true_class else "✗"
        print(f"   Cycle {cycle}: {pred['predicted_class']} ({pred['confidence']:.2%}) {correct}")

Probe Prediction Evolution

Probe 0: True class = bus
   Cycle 1: car (29.85%) ✗
   Cycle 2: bus (38.80%) ✓

Probe 1: True class = bus
   Cycle 1: bus (29.53%) ✓
   Cycle 2: bus (48.42%) ✓

Probe 2: True class = bus
   Cycle 1: bus (28.82%) ✓
   Cycle 2: bus (52.34%) ✓

Probe 3: True class = car
   Cycle 1: car (41.44%) ✓
   Cycle 2: car (33.95%) ✓

Probe 4: True class = car
   Cycle 1: car (44.50%) ✓
   Cycle 2: car (35.28%) ✓


---
## 11. Cleanup (Optional)

Remove test experiment directory if desired.

In [38]:
# List files created
print("Files created in experiment directory:")
for item in sorted(exp_dir.rglob("*")):
    if item.is_file():
        rel_path = item.relative_to(exp_dir)
        size_kb = item.stat().st_size / 1024
        print(f"   {rel_path} ({size_kb:.1f} KB)")

Files created in experiment directory:
   .state.lock (0.0 KB)
   checkpoints/best_model.pth (131129.3 KB)
   checkpoints/best_model_cycle_1.pth (131131.6 KB)
   config.yaml (0.8 KB)
   cycle_1_annotations.json (1.9 KB)
   cycle_1_confusion_matrix.npy (0.2 KB)
   cycle_1_pool_state.json (4.2 KB)
   cycle_2_confusion_matrix.npy (0.2 KB)
   experiment_state.json (16.9 KB)
   queries/cycle_1/109_Image_53.jpg (112.6 KB)
   queries/cycle_1/111_Image_88.jpg (123.9 KB)
   queries/cycle_1/194_Image_41.jpg (86.4 KB)
   queries/cycle_1/246_Image_9.jpg (309.9 KB)
   queries/cycle_1/270_Image_55.jpg (2569.1 KB)
   queries/cycle_1/44_Image_83.jpg (283.6 KB)
   queries/cycle_1/50_Image_50.jpg (146.2 KB)
   queries/cycle_1/58_Image_36.jpg (105.3 KB)
   queries/cycle_1/72_Image_13.jpg (3594.4 KB)
   queries/cycle_1/81_Image_25.png (730.0 KB)


In [40]:
# Uncomment to delete test experiment
# CLEANUP = True
CLEANUP = True

if CLEANUP:
    shutil.rmtree(exp_dir)
    print(f"✅ Deleted test experiment: {exp_dir}")
else:
    print(f"Test experiment preserved at: {exp_dir}")
    print("Set CLEANUP = True and re-run to delete.")

✅ Deleted test experiment: experiments/test_backend_20251211_164729


---
## Summary

### Tests Passed
- [ ] Configuration and state initialization
- [ ] Command infrastructure (set/clear)
- [ ] Data loading and AL components
- [ ] Probe image initialization (stratified sampling)
- [ ] Single cycle execution (all 5 phases)
- [ ] Confusion matrix saving as .npy
- [ ] Probe prediction updates per cycle
- [ ] Annotation submission and processing
- [ ] State persistence (pool state, checkpoints)
- [ ] Resume detection
- [ ] Multi-cycle operation (if run)

### Ready for Dashboard Development
If all tests pass, the backend is ready for Task 4 (Streamlit dashboard).